**ReadME**  
**V2: 100 Historial Quarters Data (5 Assets)**  

**Overview of the Entire JupyterNotebook**  
Section 1: Required Libraries  
Section 2: Data Processing
* 2.1 Single Asset Data Processing  
* 2.2 Single Asset Test Case  
* 2.3 Multi Asset Data Processing  
* 2.4 Multi Asset Test Case

Section 3: LSTM Machine Learning Estimator  
* 3.1 Pre-Processing
* 3.2 Pre-Processing Test Case
* 3.3 LSTM Model SetUp
* 3.4 LSTM Test Case

**Use Intruction**  
Please run all the sections with functions to construct the framework, Test Case Sections are optional to run

**Section 1: Required Libraries**

In [38]:
import pandas as pd
import yfinance as yf
import numpy as np

from sklearn.preprocessing import MinMaxScaler
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Dense
from tensorflow.keras.regularizers import l2

**Section 2: Data Processing**  
Section 2.1 Single Asset Data Processing

In [39]:
def get_historical_returns(ticker, start_date, end_date, frequency="monthly"):
    'Function to fetch Historical Price data and compute returns'

    data = yf.download(ticker,start=start_date, end=end_date)

    # Calculate Daily Returns
    daily_data = data.copy()
    daily_data['Return'] = daily_data['Close'].pct_change()
    daily_returns = daily_data[['Return']].dropna()

    # Calculate Monthly Returns
    monthly_data = data.copy()
    monthly_data['Return'] = monthly_data['Close']
    monthly_data = monthly_data['Return'].resample('M').last()
    monthly_returns = monthly_data.pct_change() 
    monthly_returns = monthly_returns.dropna()
    
    if frequency == "daily": return daily_returns
    if frequency == "monthly": return monthly_returns

    return monthly_data

def resample_quaterly_data(quaterly_data, target_data):
    'Repeat the quaterly available ratios to same frequency as target return'
    
    quaterly_data.index = pd.to_datetime(quaterly_data.index)
    target_data.index = pd.to_datetime(target_data.index)

    # Resample the quaterly data to daily frequency using Forward Fill
    quaterly_data.index = quaterly_data.index + pd.DateOffset(days=1)
    aligned_quaterly_data = quaterly_data.reindex(target_data.index, method='ffill')

    aligned_quaterly_data = aligned_quaterly_data.dropna()
    return aligned_quaterly_data


def load_features(path_to_file, ticker, start_date, end_date):
    'Function to Load all features for a single company'

    # Load the Excel file and read Data from the file
    file_path = path_to_file + ticker + '.xlsx'
    sheet_name = ticker + '-US'           
    data = pd.read_excel(file_path, sheet_name=sheet_name, engine='openpyxl')

    # Remove rows with any NaN values
    # Because time frame is longer, cannot apply this
    # data = data.dropna()

    # Reset the index of the DataFrame and drop the old index
    data = data.reset_index(drop=True)

    data = data.set_index('Date').T
    data.index = pd.to_datetime(data.index, format='%b \'%y')
    data.index = data.index + pd.offsets.MonthEnd()
    ratio_data = data.apply(pd.to_numeric)

    # Select a few columns
    pe_column = 'Price/Earnings'  
    pb_column = 'Price/Book Value'  
    roa_column = 'Return on Assets' 
    roe_column = 'Return on Equity ' 
    fcf_column = 'Free Cash Flow per Share'
    ratio_data = data[[pe_column, pb_column, roa_column, roe_column, fcf_column]]
    
    # Drop N/A dates
    # Removing rows with any NaN values
    ratio_data = ratio_data.dropna()

    # Process Return Data
    returns_data = get_historical_returns(ticker, start_date, end_date)
    adjusted_ratio_data = resample_quaterly_data(ratio_data, returns_data)
    features = pd.concat([adjusted_ratio_data, returns_data],axis=1)

    return features

Section 2.2 Single Asset Test Case

In [40]:
path_to_file = ''
ticker = 'MSFT'
start_date = '2000-09-30'
end_date = '2021-09-30'
msft_test = load_features(path_to_file, ticker, start_date, end_date)
msft_return_test = pd.DataFrame(msft_test['Return'])

[*********************100%%**********************]  1 of 1 completed


In [41]:
print(msft_test)

            Price/Earnings  Price/Book Value  Return on Assets  \
Date                                                             
2000-11-30       24.097222          4.971746         19.456336   
2000-12-31       24.097222          4.971746         19.456336   
2001-01-31       30.214088          6.068055         18.217195   
2001-02-28       30.214088          6.068055         18.217195   
2001-03-31       30.214088          6.068055         18.217195   
...                    ...               ...               ...   
2021-05-31       33.630867         14.345561         19.295218   
2021-06-30       33.630867         14.345561         19.295218   
2021-07-31       31.514582         13.927347         21.332801   
2021-08-31       31.514582         13.927347         21.332801   
2021-09-30       31.514582         13.927347         21.332801   

            Return on Equity   Free Cash Flow per Share    Return  
Date                                                               
2000-

In [14]:
print(msft_return_test)

              Return
Date                
2000-11-30 -0.166969
2000-12-31 -0.244009
2001-01-31  0.407781
2001-02-28 -0.033777
2001-03-31 -0.073093
...              ...
2021-05-31 -0.009914
2021-06-30  0.084989
2021-07-31  0.051717
2021-08-31  0.059563
2021-09-30 -0.059229

[251 rows x 1 columns]


Section 2.3 Multi Asset Data Processing

In [42]:
def multi_df(path_to_file, ticker_list, start_date, end_date):
    company_data = {}
    for ticker in ticker_list:
        company_data[ticker] = load_features(path_to_file, ticker, start_date, end_date)

    # Initialize a list to hold DataFrames with the new multi-index
    multi_index_dfs = []

    for company, df in company_data.items():
        # Set the company name as an additional level in the index
        df_multi_index = df.copy()
        df_multi_index['Company'] = company
        df_multi_index.set_index(['Company', df_multi_index.index], inplace=True)
        
        # Append to the list
        multi_index_dfs.append(df_multi_index)

    # Concatenate all DataFrames into a single multi-index DataFrame
    final_df = pd.concat(multi_index_dfs)

    return final_df

Section 2.4 Multi Asset Test Case
Final Output Below:  
1.multi_df: multi-index feature set (xvalue)  
2.multi_return: multi-index return set (yvalue)

In [43]:
# Loading Phase: Took a while to run this (Don't Rerun)
path_to_file = ''
stock_tickers_test1 = ["AAPL", "MSFT", "GOOG", "GOOGL", "AMZN"]
start_date = '2016-09-30'
end_date = '2021-09-30'
final_df = multi_df(path_to_file, stock_tickers_test1, start_date, end_date)

[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed


In [44]:
multi_df_test1 = final_df
multi_return_test1 = pd.DataFrame(multi_df_test1['Return'])

In [45]:
print(multi_df_test1)

                    Price/Earnings  Price/Book Value  Return on Assets  \
Company Date                                                             
AAPL    2016-10-31       13.870659          4.597652         14.482764   
        2016-11-30       13.870659          4.597652         14.482764   
        2016-12-31       13.870659          4.597652         14.482764   
        2017-01-31       16.802339          5.577686         14.294891   
        2017-02-28       16.802339          5.577686         14.294891   
...                            ...               ...               ...   
AMZN    2021-05-31       59.956081         15.162678          9.517113   
        2021-06-30       59.956081         15.162678          9.517113   
        2021-07-31       64.233702         13.814367          7.903579   
        2021-08-31       64.233702         13.814367          7.903579   
        2021-09-30       64.233702         13.814367          7.903579   

                    Return on Equity 

In [46]:
print(multi_return_test1)

                      Return
Company Date                
AAPL    2016-10-31  0.004334
        2016-11-30 -0.026599
        2016-12-31  0.047955
        2017-01-31  0.047746
        2017-02-28  0.128883
...                      ...
AMZN    2021-05-31 -0.070470
        2021-06-30  0.067355
        2021-07-31 -0.032722
        2021-08-31  0.043034
        2021-09-30 -0.048885

[300 rows x 1 columns]


**Section 3: LSTM Machine Learning Estimator**  
Section 3.1 Pre-Processing

In [47]:
def create_sequences(features, targets, seq_length):
    'Function to create sequence'
    'Need to define the sequence length: e.g. using 4 quaters to predict the next quater'

    xs = []
    ys = []

    for i in range(len(features)-seq_length):
        x = features[i:(i+seq_length)]
        y = targets.iloc[i+seq_length]
        xs.append(x)
        ys.append(y)
        
    return np.array(xs), np.array(ys)

Section 3.2 Pre-Processing Test Case

In [48]:
# Single Asset Test Case
features = msft_test
targets = msft_return_test
seq_length = 6
msft_X_test, msft_y_test = create_sequences(features, targets, seq_length)

In [49]:
print(msft_X_test)

[[[ 2.40972220e+01  4.97174600e+00  1.94563360e+01  2.46855370e+01
    1.31301600e+00 -1.66969147e-01]
  [ 2.40972220e+01  4.97174600e+00  1.94563360e+01  2.46855370e+01
    1.31301600e+00 -2.44008715e-01]
  [ 3.02140880e+01  6.06805500e+00  1.82171950e+01  2.29057140e+01
    1.46989000e+00  4.07780980e-01]
  [ 3.02140880e+01  6.06805500e+00  1.82171950e+01  2.29057140e+01
    1.46989000e+00 -3.37768680e-02]
  [ 3.02140880e+01  6.06805500e+00  1.82171950e+01  2.29057140e+01
    1.46989000e+00 -7.30932203e-02]
  [ 5.28985510e+01  8.30973200e+00  1.36502240e+01  1.74199440e+01
    1.10504100e+00  2.38857143e-01]]

 [[ 2.40972220e+01  4.97174600e+00  1.94563360e+01  2.46855370e+01
    1.31301600e+00 -2.44008715e-01]
  [ 3.02140880e+01  6.06805500e+00  1.82171950e+01  2.29057140e+01
    1.46989000e+00  4.07780980e-01]
  [ 3.02140880e+01  6.06805500e+00  1.82171950e+01  2.29057140e+01
    1.46989000e+00 -3.37768680e-02]
  [ 3.02140880e+01  6.06805500e+00  1.82171950e+01  2.29057140e+01
    

In [50]:
print(msft_y_test)

[[ 0.02110702]
 [ 0.05521827]
 [-0.09328764]
 [-0.13808737]
 [-0.1030675 ]
 [ 0.13640812]
 [ 0.1042132 ]
 [ 0.03177077]
 [-0.03833964]
 [-0.08428817]
 [ 0.03376759]
 [-0.13347708]
 [-0.02583235]
 [ 0.07444512]
 [-0.12285194]
 [ 0.02292627]
 [-0.10880196]
 [ 0.22245083]
 [ 0.07873572]
 [-0.10367544]
 [-0.08201164]
 [-0.00126417]
 [ 0.02151892]
 [ 0.05617516]
 [-0.03754396]
 [ 0.04185285]
 [ 0.03003122]
 [ 0.00416511]
 [ 0.04826541]
 [-0.05971223]
 [-0.0164499 ]
 [ 0.06456639]
 [ 0.01023013]
 [-0.04050629]
 [-0.0603091 ]
 [ 0.04813473]
 [ 0.00382703]
 [ 0.08882958]
 [-0.00245097]
 [-0.04176906]
 [ 0.01282053]
 [ 0.01157323]
 [-0.041473  ]
 [-0.00335696]
 [-0.01646701]
 [-0.04261799]
 [-0.03934816]
 [ 0.04675214]
 [ 0.01976285]
 [-0.03720927]
 [ 0.03099841]
 [ 0.06911357]
 [-0.06026295]
 [-0.00116591]
 [ 0.07704278]
 [-0.05527459]
 [ 0.07648184]
 [-0.04547065]
 [ 0.01265345]
 [-0.11245864]
 [-0.0621118 ]
 [ 0.02869756]
 [ 0.03261804]
 [ 0.06816298]
 [ 0.06420232]
 [ 0.04972573]
 [ 0.02264

In [51]:
# Multi Asset Test Case
features = multi_df_test1
targets = multi_return_test1
seq_length = 6
multi_X_test1, multi_y_test1 = create_sequences(features, targets, seq_length)

In [52]:
print(multi_X_test1)

[[[ 1.38706590e+01  4.59765200e+00  1.44827640e+01  3.46946370e+01
    2.48503400e+00  4.33434631e-03]
  [ 1.38706590e+01  4.59765200e+00  1.44827640e+01  3.46946370e+01
    2.48503400e+00 -2.65985930e-02]
  [ 1.38706590e+01  4.59765200e+00  1.44827640e+01  3.46946370e+01
    2.48503400e+00  4.79551503e-02]
  [ 1.68023390e+01  5.57768600e+00  1.42948910e+01  3.45733520e+01
    2.52979700e+00  4.77464928e-02]
  [ 1.68023390e+01  5.57768600e+00  1.42948910e+01  3.45733520e+01
    2.52979700e+00  1.28883455e-01]
  [ 1.68023390e+01  5.57768600e+00  1.42948910e+01  3.45733520e+01
    2.52979700e+00  4.86896701e-02]]

 [[ 1.38706590e+01  4.59765200e+00  1.44827640e+01  3.46946370e+01
    2.48503400e+00 -2.65985930e-02]
  [ 1.38706590e+01  4.59765200e+00  1.44827640e+01  3.46946370e+01
    2.48503400e+00  4.79551503e-02]
  [ 1.68023390e+01  5.57768600e+00  1.42948910e+01  3.45733520e+01
    2.52979700e+00  4.77464928e-02]
  [ 1.68023390e+01  5.57768600e+00  1.42948910e+01  3.45733520e+01
    

In [53]:
print(multi_y_test1)

[[-6.96767741e-05]
 [ 6.34180369e-02]
 [-5.72138685e-02]
 [ 3.27037308e-02]
 [ 1.02669298e-01]
 [-6.02439322e-02]
 [ 9.68076735e-02]
 [ 1.66233609e-02]
 [-1.52459138e-02]
 [-1.06364303e-02]
 [ 6.38475955e-02]
 [-5.80507333e-02]
 [-1.50196942e-02]
 [ 1.30763653e-01]
 [-9.41828305e-03]
 [ 2.79833216e-02]
 [ 1.96226880e-01]
 [-8.30294491e-03]
 [-3.04775614e-02]
 [-1.84044589e-01]
 [-1.16698377e-01]
 [ 5.51540297e-02]
 [ 4.03147762e-02]
 [ 9.70257213e-02]
 [ 5.64359115e-02]
 [-1.27572587e-01]
 [ 1.30519163e-01]
 [ 7.63944789e-02]
 [-2.01839463e-02]
 [ 7.29615566e-02]
 [ 1.10684436e-01]
 [ 7.43286939e-02]
 [ 9.87838874e-02]
 [ 5.40099309e-02]
 [-1.16797594e-01]
 [-6.97614614e-02]
 [ 1.55373768e-01]
 [ 8.21647912e-02]
 [ 1.47386252e-01]
 [ 1.65131641e-01]
 [ 2.14379735e-01]
 [-1.02526321e-01]
 [-6.00120637e-02]
 [ 9.36064889e-02]
 [ 1.14573700e-01]
 [-5.50151265e-03]
 [-8.10852079e-02]
 [ 7.33959569e-03]
 [ 7.62178066e-02]
 [-5.21071486e-02]
 [ 9.91092693e-02]
 [ 6.49824289e-02]
 [ 4.0929666

Section 3.3 LSTM Model SetUp

In [54]:
# LSTM Model Set Up

# Model architecture
model = tf.keras.Sequential([
    LSTM(512, return_sequences=True),
    Dropout(0.02),
    LSTM(256, return_sequences=True),
    LSTM(128),
    Dense(1, activation='linear', kernel_regularizer=l2(0.0005))
])

# Compile the model
optimizer = tf.keras.optimizers.Adam(learning_rate=0.005)
model.compile(optimizer=optimizer, loss='mean_squared_error')

Section 3.4 LSTM Test Case

In [55]:
# Variable in Use and Constant Define
X = multi_X_test1
y = multi_y_test1
train_ratio = 0.8
num_epoch = 100
num_batch = 12

# Split data into training and validation sets

train_size = int(len(X) * train_ratio)
X_train, X_vali = X[:train_size], X[train_size:]
y_train, y_vali = y[:train_size], y[train_size:]

# Train the model
model.fit(X_train, y_train, epochs=num_epoch, batch_size=num_batch, validation_data=(X_vali, y_vali))


Epoch 1/100
20/20 [==============================] - 5s 53ms/step - loss: 0.8474 - val_loss: 0.0134
Epoch 2/100
20/20 [==============================] - 0s 18ms/step - loss: 0.0057 - val_loss: 0.0089
Epoch 3/100
20/20 [==============================] - 0s 17ms/step - loss: 0.0053 - val_loss: 0.0072
Epoch 4/100
20/20 [==============================] - 0s 19ms/step - loss: 0.0050 - val_loss: 0.0074
Epoch 5/100
20/20 [==============================] - 0s 18ms/step - loss: 0.0051 - val_loss: 0.0072
Epoch 6/100
20/20 [==============================] - 0s 17ms/step - loss: 0.0051 - val_loss: 0.0076
Epoch 7/100
20/20 [==============================] - 0s 17ms/step - loss: 0.0050 - val_loss: 0.0082
Epoch 8/100
20/20 [==============================] - 0s 17ms/step - loss: 0.0054 - val_loss: 0.0073
Epoch 9/100
20/20 [==============================] - 0s 19ms/step - loss: 0.0052 - val_loss: 0.0079
Epoch 10/100
20/20 [==============================] - 0s 17ms/step - loss: 0.0050 - val_loss: 0.0072

In [56]:
# Evaluate the model
X_test, y_test = X_vali, y_vali

loss = model.evaluate(X_test, y_test)
print(f"Mean Squared Error: {loss}")

2/2 [==============================] - 0s 8ms/step - loss: 0.0071
Mean Squared Error: 0.007140048313885927


In [57]:
# Make predictions
X_pred, y_pred = X_vali, y_vali

predictions = model.predict(X_pred)
print(predictions)

2/2 [==============================] - 1s 6ms/step
[[0.02940229]
 [0.02809858]
 [0.02226913]
 [0.01510433]
 [0.01264077]
 [0.00815923]
 [0.00782416]
 [0.00794401]
 [0.00795587]
 [0.00783764]
 [0.00768716]
 [0.00764933]
 [0.0079791 ]
 [0.00799425]
 [0.00808589]
 [0.00820675]
 [0.00813502]
 [0.00812957]
 [0.00876658]
 [0.00936275]
 [0.01122578]
 [0.01322104]
 [0.01417473]
 [0.0153437 ]
 [0.01615275]
 [0.01672962]
 [0.01723771]
 [0.01745745]
 [0.01766603]
 [0.01786542]
 [0.01776106]
 [0.01780926]
 [0.01769229]
 [0.01711141]
 [0.01703589]
 [0.01695171]
 [0.01681012]
 [0.01675043]
 [0.0166691 ]
 [0.0165188 ]
 [0.01650742]
 [0.01637259]
 [0.0165393 ]
 [0.01639903]
 [0.01606508]
 [0.0162975 ]
 [0.01630692]
 [0.01652589]
 [0.01683212]
 [0.01704303]
 [0.01731065]
 [0.01265658]
 [0.0125869 ]
 [0.01438319]
 [0.01520762]
 [0.01573483]
 [0.01560637]
 [0.01699329]
 [0.01724561]]


In [58]:
print(y_pred)

[[-0.0496949 ]
 [-0.00093262]
 [ 0.09816368]
 [ 0.02618155]
 [ 0.04911012]
 [ 0.04337087]
 [ 0.07527646]
 [-0.02676394]
 [ 0.02043385]
 [-0.00726885]
 [-0.01963079]
 [ 0.14971652]
 [ 0.06466238]
 [-0.00618657]
 [ 0.24063898]
 [ 0.04242906]
 [-0.04304937]
 [ 0.0820748 ]
 [ 0.04053941]
 [ 0.04306519]
 [ 0.04567601]
 [ 0.13236448]
 [-0.00482431]
 [-0.20219175]
 [ 0.05767175]
 [-0.1113497 ]
 [ 0.14431709]
 [-0.04590598]
 [ 0.08593571]
 [ 0.08185875]
 [-0.0786132 ]
 [ 0.06679175]
 [-0.01417918]
 [-0.04847382]
 [-0.02273274]
 [ 0.0234747 ]
 [ 0.0135873 ]
 [ 0.02612169]
 [ 0.0870638 ]
 [-0.06221372]
 [ 0.03502057]
 [ 0.26890012]
 [-0.01278494]
 [ 0.12956673]
 [ 0.14711362]
 [ 0.09046103]
 [-0.08757859]
 [-0.03575409]
 [ 0.04343987]
 [ 0.02805838]
 [-0.01557601]
 [-0.03532841]
 [ 0.00037178]
 [ 0.12066274]
 [-0.07047026]
 [ 0.06735499]
 [-0.03272228]
 [ 0.04303417]
 [-0.04888515]]
